<a href="https://colab.research.google.com/github/apk41910/gas_calculator/blob/main/gas_calculator_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
R = 0.08206  # L·atm/(mol·K)

from scipy.optimize import brentq

def calc_P(V, n, T):
    P=n*R*T/V
    return P

def calc_V(P, n, T):
    V=n*R*T/P
    return V

def calc_n(P, V, T):
    n=P*V/R/T
    return n

def calc_T(P, V, n):
    T=P*V/n/R
    return T


def to_kelvin(value, unit):
    if unit == "k":
        return value
    elif unit == "c":
        return value + 273.15
    elif unit == "f":
        return (value - 32) * 5/9 + 273.15
    elif unit == "r":
        return value * 5/9


def to_base(var):
    number, unit = values[var]

    if var == "P":
        return number * to_atm[unit]
    elif var == "V":
        return number * to_L[unit]
    elif var == "n":
        return number * to_mol[unit]
    elif var == "T":
        return to_kelvin(number, unit)



to_atm = {
    "atm":  1,
    "pa":   1/101325,
    "kpa":  1/101.325,
    "mpa":  1/0.101325,
    "bar":  1/1.01325,
    "mmhg": 1/760,
    "torr": 1/760,
    "psi":  1/14.696,
  }

# 부피 → L
to_L = {
    "l":   1,
    "ml":  0.001,
    "cm3": 0.001,
    "m3":  1000,
    "ft3": 28.317,
    "gal": 3.7854,
}

# 몰수 → mol
to_mol = {
    "mol":  1,
    "mmol": 0.001,
    "kmol": 1000,
}


to_T = {
    "k": 1,
    "c": 1,
    "f": 1,
    "r": 1
}


# Tc, Pc: 임계온도(K), 임계압력(atm)
# omega: 이심인자(acentric factor)
# M: 분자량(g/mol)
# antoine: (A, B, C) — log10(P/mmHg) = A - B/(C + T), T는 °C
# antoine_range: antoine 계수가 유효한 온도 범위 (°C)

substances = {
    "co2":  {"name": "CO2",  "M": 44.01, "Tc": 304.2, "Pc": 72.9,  "omega": 0.225,
             "antoine": (7.5788, 865.71, 273.48),    "antoine_range": (-119, -69)},

    "n2":   {"name": "N2",   "M": 28.01, "Tc": 126.2, "Pc": 33.5,  "omega": 0.040,
             "antoine": (6.49457, 255.68, 266.55),   "antoine_range": (-210, -184)},

    "o2":   {"name": "O2",   "M": 32.00, "Tc": 154.6, "Pc": 49.8,  "omega": 0.022,
             "antoine": (6.69144, 319.013, 266.697), "antoine_range": (-219, -183)},

    "ch4":  {"name": "CH4",  "M": 16.04, "Tc": 190.6, "Pc": 45.4,  "omega": 0.011,
             "antoine": (6.61184, 389.93, 266.00),   "antoine_range": (-181, -152)},

    "h2o":  {"name": "H2O",  "M": 18.02, "Tc": 647.1, "Pc": 217.8, "omega": 0.345,
             "antoine": (8.07131, 1730.63, 233.426), "antoine_range": (1, 100)},

    "nh3":  {"name": "NH3",  "M": 17.03, "Tc": 405.5, "Pc": 111.3, "omega": 0.250,
             "antoine": (7.36050, 926.132, 240.17),  "antoine_range": (-83, 60)},

    "h2":   {"name": "H2",   "M": 2.016, "Tc": 33.2,  "Pc": 12.8,  "omega": -0.216,
             "antoine": (5.92088, 71.6153, 276.34),  "antoine_range": (-259, -253)},

    "c2h6": {"name": "C2H6", "M": 30.07, "Tc": 305.3, "Pc": 48.1,  "omega": 0.099,
             "antoine": (6.83452, 663.70, 256.47),   "antoine_range": (-143, -75)},
}



In [3]:
class EOS:
    name = "EOS"

    def __init__(self, sub):
        self.sub = sub

    def ab(self):
        raise NotImplementedError

    def alpha(self, T):
        return 1.0

    def pressure(self, Vm, T):
        raise NotImplementedError

    def solve_Vm(self, P_target, T):
        a, b = self.ab()
        low = max(b * 1.001, 1e-6)
        return brentq(lambda Vm: self.pressure(Vm, T) - P_target, low, 1000.0)

    def solve_T(self, P_target, Vm):
        return brentq(lambda T: self.pressure(Vm, T) - P_target, 1, 5000)


class Ideal(EOS):
    name = "Ideal"

    def ab(self):
        return 0.0, 0.0

    def pressure(self, Vm, T):
        return R*T/Vm

class VdW(EOS):
    name = "vdW"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 27*R**2*Tc**2/(64*Pc)
        b = R*Tc/(8*Pc)
        return a, b

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a/Vm**2
class RK(EOS):
    name = "RK"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        c = 2 ** (1/3)
        a = 1 / (9 * (c - 1)) * R**2 * Tc**2.5 / Pc
        b = (c - 1) / 3 * R * Tc / Pc
        return a, b

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a/(T**0.5 * Vm * (Vm + b))



class SRK(EOS):
    name = "SRK"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 0.42748*R**2*Tc**2/Pc
        b = 0.08664*R*Tc/Pc
        return a, b

    def alpha(self, T):
        Tc = self.sub["Tc"]
        omega = self.sub["omega"]
        m = 0.480 + 1.574*omega - 0.176*omega**2
        return (1 + m*(1 - (T/Tc)**0.5))**2

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a*self.alpha(T)/(Vm*(Vm + b))

class PR(EOS):
    name = "PR"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 0.45724*R**2*Tc**2/Pc
        b = 0.07780*R*Tc/Pc
        return a, b

    def alpha(self, T):
        Tc = self.sub["Tc"]
        omega = self.sub["omega"]
        kappa = 0.37464 + 1.54226*omega - 0.26992*omega**2
        return (1 + kappa*(1 - (T/Tc)**0.5))**2

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a*self.alpha(T)/(Vm*(Vm + b) + b*(Vm - b))



import math

def vapor_pressure(T, sub):
    """T(K)에서의 증기압(atm)과 사용한 방법. 계산 불가면 (None, None)"""
    Tc, Pc, omega = sub["Tc"], sub["Pc"], sub["omega"]
    Tr = T / Tc

    # 조건 1: 임계온도 이상이면 증기압 자체가 없음
    if Tr >= 1:
        return None, None

    # 조건 2: Antoine 계수가 있고 범위 안이면 실측 기반 우선
    if "antoine" in sub:
        A, B, C = sub["antoine"]
        Tmin, Tmax = sub["antoine_range"]
        T_celsius = T - 273.15
        if Tmin <= T_celsius <= Tmax:
            return 10 ** (A - B/(C + T_celsius)) / 760, "Antoine"

    # 조건 3: 너무 저온이면 상관식도 못 씀
    if Tr < 0.3:
        return None, None

    # 조건 4: 그 외 → Ambrose-Walton
    tau = 1 - Tr
    f0 = (-5.97616*tau + 1.29874*tau**1.5 - 0.60394*tau**2.5 - 1.06841*tau**5) / Tr
    f1 = (-5.03365*tau + 1.11505*tau**1.5 - 5.41217*tau**2.5 - 7.46628*tau**5) / Tr
    f2 = (-0.64771*tau + 2.41539*tau**1.5 - 4.26979*tau**2.5 + 3.25259*tau**5) / Tr

    return Pc * math.exp(f0 + omega*f1 + omega**2 * f2), "Ambrose-Walton"


def check_phase(P, T, sub):
    """(상, 사용한 증기압 식) 을 return"""
    Tc, Pc = sub["Tc"], sub["Pc"]

    # 조건 1: 임계온도 위
    if T > Tc:
        return ("초임계" if P > Pc else "기체"), None

    # 조건 2: 증기압과 비교
    Psat, method = vapor_pressure(T, sub)

    if Psat is None:
        return "판별불가", None

    return ("기체" if P < Psat else "액체"), method

def print_result(label, value, unit, phase=None):
    """계산 결과 한 줄 출력. value가 None이면 '해 없음'"""
    if value is None:
        print(f"  {label:<6} {'해 없음':>12}   (기체 아님)")
        return

    line = f"  {label:<6} {value:>12.4f} {unit:<4}"
    if phase:
        line += f"   ({phase})"
    print(line)


def print_phase(P, T, sub):
    """계산 전 상 경고. (상, 사용한 증기압 식) return"""
    phase, method = check_phase(P, T, sub)
    if phase == "액체":
        print("  ※ 이 조건에서는 액체입니다. 아래 결과는 신뢰할 수 없습니다.")
    return phase, method

In [13]:
while True:
    name = input("물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): ").lower()

    if name == "q":
        break

    if name not in substances:
        print("등록되지 않은 물질입니다.")
        continue

    sub = substances[name]
    print(f"{sub['name']} 선택됨")

    print("사용 가능한 단위")
    print("  압력: atm, kPa, bar, mmHg, psi ...")
    print("  부피: L, mL, m3 ...")
    print("  몰수: mol, mmol, kmol")
    print("  온도: K, C, F")

    eos_list = [Ideal(sub), VdW(sub), RK(sub), SRK(sub), PR(sub)]

    while True:
        text = input(f"[{sub['name']}] 값 3개 입력 (b: 물질 변경) > ")

        if text == "b":
            break

        values = {}
        has_error = False

        for item in text.split(","):
            try:
                number, unit = item.split()
                number = float(number)
                unit = unit.lower()

                if unit in to_atm:
                    values["P"] = (number, unit)
                elif unit in to_L:
                    values["V"] = (number, unit)
                elif unit in to_mol:
                    values["n"] = (number, unit)
                elif unit in to_T:
                    values["T"] = (number, unit)
                else:
                    print("모르는 단위입니다:", unit)
            except ValueError:
                print("입력 형식을 확인해주세요. 예: 200 kPa, 2 mol, 300 K")
                has_error = True
                break

        if has_error:
            continue

        missing = [v for v in ["P", "V", "n", "T"] if v not in values]

        if len(missing) != 1:
            print("값 3개를 입력해주세요.")
            continue

        target = missing[0]
        print("{}를 구하겠습니다".format(target))

        if target == "P":
            V, n, T = to_base("V"), to_base("n"), to_base("T")
            Vm = V / n

            for eos in eos_list:
                P = eos.pressure(Vm, T)
                phase, method = check_phase(P, T, sub)
                label = f"{phase}, {method}" if method else phase
                print_result(eos.name, P, "atm", label)

        elif target == "V":
            P, n, T = to_base("P"), to_base("n"), to_base("T")
            phase, method = print_phase(P, T, sub)
            label = f"{phase}, {method}" if method else phase

            for eos in eos_list:
                try:
                    print_result(eos.name, eos.solve_Vm(P, T) * n, "L", label)
                except ValueError:
                    print_result(eos.name, None, "L")

        elif target == "n":
            P, V, T = to_base("P"), to_base("V"), to_base("T")
            phase, method = print_phase(P, T, sub)
            label = f"{phase}, {method}" if method else phase

            for eos in eos_list:
                try:
                    print_result(eos.name, V / eos.solve_Vm(P, T), "mol", label)
                except ValueError:
                    print_result(eos.name, None, "mol")

        elif target == "T":
            P, V, n = to_base("P"), to_base("V"), to_base("n")
            Vm = V / n

            for eos in eos_list:
                try:
                    T = eos.solve_T(P, Vm)
                    phase, method = check_phase(P, T, sub)
                    label = f"{phase}, {method}" if method else phase
                    print_result(eos.name, T, "K", label)
                except ValueError:
                    print_result(eos.name, None, "K")

물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): co2
CO2 선택됨
사용 가능한 단위
  압력: atm, kPa, bar, mmHg, psi ...
  부피: L, mL, m3 ...
  몰수: mol, mmol, kmol
  온도: K, C, F
[CO2] 값 3개 입력 (b: 물질 변경) > 200 kPa, 2 mol, 300 K
V를 구하겠습니다
  Ideal       24.9442 L      (기체, Ambrose-Walton)
  vdW         24.7354 L      (기체, Ambrose-Walton)
  RK          24.7031 L      (기체, Ambrose-Walton)
  SRK         24.7018 L      (기체, Ambrose-Walton)
  PR          24.6754 L      (기체, Ambrose-Walton)
[CO2] 값 3개 입력 (b: 물질 변경) > b
물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): q


In [14]:
sub = substances["co2"]
T = 300
eos_list = [Ideal(sub), VdW(sub), RK(sub), SRK(sub), PR(sub)]


print(f"{'P(atm)':>8}", end="")
for eos in eos_list:
    print(f"{eos.name:>10}", end="")
print()


for P in range(10, 201, 10):
    print(f"{P:>8}", end="")
    for eos in eos_list:
        try:
            Vm = eos.solve_Vm(P, T)
            Z = P * Vm / (R * T)
            print(f"{Z:>10.4f}", end="")
        except ValueError:
            print(f"{'-':>10}", end="")
    print()

  P(atm)     Ideal       vdW        RK       SRK        PR
      10    1.0000    0.9563    0.9497    0.9494    0.9442
      20    1.0000    0.9088    0.8956    0.8950    0.8846
      30    1.0000    0.8564    0.8364    0.8353    0.8202
      40    1.0000    0.7970    0.7700    0.7681    0.7485
      50    1.0000    0.7264    0.6917    0.6888    0.6651
      60    1.0000    0.6337    0.5892    0.5835    0.5563
      70    1.0000    0.2826    0.2263    0.2193    0.1974
      80    1.0000    0.2839    0.2320    0.2284    0.2054
      90    1.0000    0.3020    0.2464    0.2436    0.2189
     100    1.0000    0.3232    0.2629    0.2604    0.2339
     110    1.0000    0.3455    0.2801    0.2779    0.2495
     120    1.0000    0.3682    0.2976    0.2955    0.2653
     130    1.0000    0.3912    0.3153    0.3133    0.2812
     140    1.0000    0.4142    0.3330    0.3310    0.2970
     150    1.0000    0.4372    0.3506    0.3487    0.3128
     160    1.0000    0.4601    0.3681    0.3663    0.32

In [15]:
import matplotlib.pyplot as plt
import numpy as np

# 식 이름 → 클래스 (위젯에서 문자열을 받을 때 사용)
EOS_MAP = {"Ideal": Ideal, "VdW": VdW, "RK": RK, "SRK": SRK, "PR": PR}


def plot_Z(sub, T, Pmax=200, step=5):
    """한 온도에서 다섯 식의 압축인자 Z 비교"""
    eos_list = [cls(sub) for cls in EOS_MAP.values()]
    pressures = list(range(step, Pmax + 1, step))

    for eos in eos_list:
        zs = []
        for P in pressures:
            try:
                Vm = eos.solve_Vm(P, T)
                zs.append(P * Vm / (R * T))
            except ValueError:
                zs.append(None)
        plt.plot(pressures, zs, label=eos.name)

    plt.xlabel("P (atm)")
    plt.ylabel("Z = PVm/RT")
    plt.title(f"{sub['name']} at {T} K")
    plt.ylim(0, 1.2)


def plot_Z_temps(sub, temperatures, eos_class=PR, Pmax=200, step=5):
    """여러 온도에서 한 식의 Z 곡선"""
    eos = eos_class(sub)
    pressures = list(range(step, Pmax + 1, step))

    for T in temperatures:
        zs = []
        for P in pressures:
            try:
                Vm = eos.solve_Vm(P, T)
                zs.append(P * Vm / (R * T))
            except ValueError:
                zs.append(None)
        plt.plot(pressures, zs, label=f"{T} K")

    plt.xlabel("P (atm)")
    plt.ylabel("Z = PVm/RT")
    plt.title(f"{sub['name']} ({eos.name})")
    plt.ylim(0, 1.2)


def plot_PV(sub, temperatures, eos_class=VdW):
    """P-V 등온선. solve 없이 Vm을 직접 주므로 S자 구간도 그려짐"""
    eos = eos_class(sub)
    a, b = eos.ab()
    vms = [b * 1.05 * (1.05 ** i) for i in range(150)]

    for T in temperatures:
        ps = [eos.pressure(Vm, T) for Vm in vms]
        plt.plot(vms, ps, label=f"{T} K")

    plt.xscale("log")
    plt.ylim(0, sub["Pc"] * 2)
    plt.xlabel("Vm (L/mol)")
    plt.ylabel("P (atm)")
    plt.title(f"{sub['name']} P-V isotherms ({eos.name})")


def plot_generalized(sub_names, Tr_list=(1.0, 1.2, 1.5, 2.0)):
    """환산 좌표(Pr, Tr)로 그린 일반화 압축인자 차트"""
    for Tr in Tr_list:
        for name in sub_names:
            sub = substances[name]
            T = Tr * sub["Tc"]
            eos = PR(sub)
            prs, zs = [], []
            for i in range(1, 71):
                Pr = i / 10
                P = Pr * sub["Pc"]
                try:
                    Vm = eos.solve_Vm(P, T)
                    prs.append(Pr)
                    zs.append(P * Vm / (R * T))
                except ValueError:
                    pass
            plt.plot(prs, zs, label=f"{sub['name']} Tr={Tr}")

    plt.xlabel("Pr = P/Pc")
    plt.ylabel("Z")
    plt.title("Generalized compressibility chart (PR)")
    plt.ylim(0, 1.2)


def plot_error_map(sub, T_range, P_range, n=40):
    """이상기체 가정의 오차 |Z-1| 을 T-P 평면에 표시"""
    Ts = np.linspace(*T_range, n)
    Ps = np.linspace(*P_range, n)
    err = np.zeros((n, n))
    eos = PR(sub)

    for i, T in enumerate(Ts):
        for j, P in enumerate(Ps):
            try:
                Vm = eos.solve_Vm(P, T)
                err[i, j] = abs(P * Vm / (R * T) - 1) * 100
            except ValueError:
                err[i, j] = np.nan

    plt.contourf(Ps, Ts, err, levels=20, cmap="RdYlGn_r")
    plt.colorbar(label="|Z-1| (%)")
    plt.xlabel("P (atm)")
    plt.ylabel("T (K)")
    plt.title(f"{sub['name']}: deviation from ideal gas (PR)")

from ipywidgets import interact, interact_manual, IntSlider, FloatSlider, Dropdown, SelectMultiple

SUB_NAMES = list(substances.keys())
EOS_NAMES = ["VdW", "RK", "SRK", "PR"]


In [16]:
# 1. 식 비교 (한 온도에서 다섯 식)

@interact(
    name=Dropdown(options=SUB_NAMES, value="co2", description="물질"),
    T=IntSlider(min=100, max=700, step=10, value=300, description="T (K)"),
    Pmax=IntSlider(min=50, max=500, step=50, value=200, description="P max")
)
def show_Z(name, T, Pmax):
    plot_Z(substances[name], T, Pmax)
    plt.legend()
    plt.grid(True)
    plt.show()

interactive(children=(Dropdown(description='물질', options=('co2', 'n2', 'o2', 'ch4', 'h2o', 'nh3', 'h2', 'c2h6'…

In [8]:
# 2. 온도 비교 (한 식으로 여러 온도)
@interact(
    name=Dropdown(options=SUB_NAMES, value="co2", description="물질"),
    eos_name=Dropdown(options=EOS_NAMES, value="PR", description="식"),
    T1=IntSlider(min=100, max=700, step=10, value=250, description="T1 (K)"),
    T2=IntSlider(min=100, max=700, step=10, value=300, description="T2 (K)"),
    T3=IntSlider(min=100, max=700, step=10, value=350, description="T3 (K)"),
    Pmax=IntSlider(min=50, max=500, step=50, value=200, description="P max")
)
def show_Z_temps(name, eos_name, T1, T2, T3, Pmax):
    plot_Z_temps(substances[name], [T1, T2, T3], EOS_MAP[eos_name], Pmax)
    plt.legend()
    plt.grid(True)
    plt.show()

interactive(children=(Dropdown(description='물질', options=('co2', 'n2', 'o2', 'ch4', 'h2o', 'nh3', 'h2', 'c2h6'…

In [9]:
# 3. P-V 등온선
@interact(
    name=Dropdown(options=SUB_NAMES, value="co2", description="물질"),
    eos_name=Dropdown(options=EOS_NAMES, value="VdW", description="식"),
    T1=IntSlider(min=100, max=700, step=10, value=250, description="T1 (K)"),
    T2=IntSlider(min=100, max=700, step=10, value=290, description="T2 (K)"),
    T3=IntSlider(min=100, max=700, step=10, value=320, description="T3 (K)")
)
def show_PV(name, eos_name, T1, T2, T3):
    plot_PV(substances[name], [T1, T2, T3], EOS_MAP[eos_name])
    plt.legend()
    plt.grid(True)
    plt.show()

interactive(children=(Dropdown(description='물질', options=('co2', 'n2', 'o2', 'ch4', 'h2o', 'nh3', 'h2', 'c2h6'…

In [10]:
# 4. 일반화 압축인자 차트
@interact(
    names=SelectMultiple(options=SUB_NAMES, value=("co2", "n2", "ch4"),
                         description="물질", rows=6),
    Tr=FloatSlider(min=1.0, max=3.0, step=0.1, value=1.2, description="Tr")
)
def show_gen(names, Tr):
    plot_generalized(list(names), [Tr])
    plt.legend(fontsize=8)
    plt.grid(True)
    plt.show()

interactive(children=(SelectMultiple(description='물질', index=(0, 1, 3), options=('co2', 'n2', 'o2', 'ch4', 'h2…

In [12]:
from ipywidgets import interact_manual, IntSlider, Dropdown

# 5. 오차 지도
@interact_manual(
    name=Dropdown(options=SUB_NAMES, value="co2", description="물질"),
    Tmax=IntSlider(min=300, max=1000, step=50, value=600, description="T max"),
    Pmax=IntSlider(min=50, max=500, step=50, value=200, description="P max"),
    n=IntSlider(min=10, max=60, step=10, value=20, description="해상도")
)
def show_map(name, Tmax, Pmax, n):
    sub = substances[name]
    plot_error_map(sub, (sub["Tc"] * 0.8, Tmax), (1, Pmax), n)
    plt.show()

interactive(children=(Dropdown(description='물질', options=('co2', 'n2', 'o2', 'ch4', 'h2o', 'nh3', 'h2', 'c2h6'…